# 00 — Train LSTM Embedder (Daily Candles)

Обучение LSTM embedder с подбором гиперпараметров через Optuna (метрика: IC).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.abspath("../../src"))

import numpy as np
import optuna

from data.dataloader import Dataloader
from data.tickers import TICKERS
from embeddings.LSTM_embedder import LSTMEmbedder
from evaluation.metrics import information_coefficient

INTERVAL = "daily"
WINDOW_SIZE = 60
STEP = 60
HORIZON = 1
RANDOM_STATE = 42

FEATURE_COLS = [
    "open", "high", "low", "close", "volume",
    "sma5", "sma20", "ema12", "ema26", "close_sma20",
    "macd", "macd_signal", "macd_hist", "rsi14",
    "bb_pct", "bb_bw", "atr14", "obv",
]

N_TRIALS = 50
LR = 1e-3
CHECKPOINT_PATH = "../../models/lstm_embedder_daily.pth"

np.random.seed(RANDOM_STATE)

In [ ]:
# Load data
loader = Dataloader()

dataset = loader.cut_on_windows(
    tickers=TICKERS,
    interval=INTERVAL,
    seq_len=WINDOW_SIZE,
    step_size=STEP,
    horizon=HORIZON,
    feature_cols=FEATURE_COLS,
    add_indicators=True,
)

X_train, y_train, X_val, y_val, X_test, y_test = loader.concat_splits(dataset)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

In [ ]:
# Optuna optimization
import torch

def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 32, 128, step=32)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    epochs = trial.suggest_int("epochs", 20, 50)
    scheduler = trial.suggest_categorical("scheduler", ["cosine", "reduce_on_plateau", "step", "none"])
    
    model = LSTMEmbedder(
        input_size=X_train.shape[2],
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    )
    
    model.fit(
        X_train, y_train,
        X_val=X_val, y_val=y_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=LR,
        clip_grad=1.0,
        verbose=False,
        scheduler=scheduler,
    )
    
    model.eval()
    with torch.no_grad():
        device = next(model.parameters()).device
        X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
        y_pred = model(X_val_t).cpu().numpy()
    
    ic = information_coefficient(y_val, y_pred)
    return ic

study = optuna.create_study(direction="maximize", study_name="lstm_embedder_daily")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial:")
print(f"  IC: {study.best_value:.6f}")
print(f"  Params: {study.best_params}")

In [ ]:
# Train final model with best params
best_params = study.best_params

lstm_emb = LSTMEmbedder(
    input_size=X_train.shape[2],
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"],
    dropout=best_params["dropout"],
)

print(f"Training final model with best params:")
print(f"  hidden_size: {best_params['hidden_size']}")
print(f"  num_layers: {best_params['num_layers']}")
print(f"  dropout: {best_params['dropout']:.3f}")
print(f"  batch_size: {best_params['batch_size']}")
print(f"  epochs: {best_params['epochs']}")
print(f"  scheduler: {best_params['scheduler']}")

lstm_emb.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    lr=LR,
    clip_grad=1.0,
    verbose=True,
    scheduler=best_params["scheduler"],
)

lstm_emb.plot_loss()

os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
lstm_emb.save(CHECKPOINT_PATH)
print(f"\nModel saved to: {CHECKPOINT_PATH}")

In [ ]:
# Evaluate final model IC
import torch

lstm_emb.eval()
with torch.no_grad():
    device = next(lstm_emb.parameters()).device
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_pred = lstm_emb(X_val_t).cpu().numpy()
    
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_pred = lstm_emb(X_test_t).cpu().numpy()

ic_val = information_coefficient(y_val, y_val_pred)
ic_test = information_coefficient(y_test, y_test_pred)

print(f"Final model IC:")
print(f"  Validation: {ic_val:.6f}")
print(f"  Test:       {ic_test:.6f}")